# Synthetic Floor 7-Stage - GPU Blender Renderer (Colab)

Renders the 7 progressive construction stages with Blender Cycles on a Colab GPU.

**Production features:**
- **Google Drive**: outputs are mirrored to Drive after every stage (and by a
  background daemon), so a disconnect / runtime reset never loses work.
- **Resume**: re-run with the same run name and finished stages are skipped;
  partial progress is restored from Drive automatically. Works from another
  device / Drive account too - just point at the same Drive folder.
- **Correct geometry**: the renderer re-orients the imported mesh into the
  authored Z-up frame, fixing the old 'bright scene / nothing but light' bug
  where the room was rotated out of the camera's view.

Set the runtime to **GPU** first: *Runtime -> Change runtime type -> T4 (or A100)*.


## 1. Verify GPU, mount Drive, clone the repo


In [ ]:
import os, sys, subprocess

# Pick a stable run name. Re-use the SAME name to resume a previous run
# (even in a new session or on another machine sharing this Drive).
RUN_NAME = 'demo'

try:
    print('GPU:', subprocess.check_output(['nvidia-smi','-L']).decode().strip())
except Exception:
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4/A100 (CPU will be very slow).')

# Mount Google Drive (idempotent).
DRIVE_ROOT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = f'/content/drive/MyDrive/MyCon_Colab/synthetic_floor_7stage/{RUN_NAME}'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('Drive output folder:', DRIVE_ROOT)
except Exception as exc:
    print('Drive not available (not on Colab?):', exc)

if not os.path.isdir('/content/MyCon'):
    subprocess.check_call(['git','clone','--depth','1','https://github.com/sayyedalimrj/MyCon.git','/content/MyCon'])
else:
    subprocess.call(['git','-C','/content/MyCon','pull','--ff-only'])
os.chdir('/content/MyCon/repository')
print('cwd:', os.getcwd())


## 2. Install Blender 4.2 LTS


In [ ]:
%%bash
set -e
BLENDER_VERSION="4.2.3"
BLENDER_TAR="blender-${BLENDER_VERSION}-linux-x64.tar.xz"
BLENDER_URL="https://download.blender.org/release/Blender4.2/${BLENDER_TAR}"
if [ ! -f "/content/blender/blender" ]; then
  echo "Downloading Blender ${BLENDER_VERSION} ..."
  wget -q "${BLENDER_URL}" -O /tmp/blender.tar.xz
  mkdir -p /content/blender
  tar -xf /tmp/blender.tar.xz -C /content/blender --strip-components=1
  rm /tmp/blender.tar.xz
  echo "Blender installed."
else
  echo "Blender already installed."
fi
apt-get -qq install -y libxrender1 libxi6 libxkbcommon0 libsm6 libxxf86vm1 libgl1 libegl1 libxfixes3 > /dev/null 2>&1 || true
/content/blender/blender --version 2>&1 | head -2


## 3. Install Python dependencies


In [ ]:
!pip install -q numpy Pillow PyYAML trimesh imageio imageio-ffmpeg ifcopenshell scipy


## 4. Smoke render - stage 7, debug preset (with Drive sync + resume)

`--mount-drive --drive-root` mirror every output to Drive; `--resume` skips
stages already complete on Drive. `--strict-render` fails if a frame comes out
blank (the brightness/coverage guard).


In [ ]:
!PYTHONPATH=examples/synthetic_floor_7stage/src \
    python3 examples/synthetic_floor_7stage/scripts/run_blender_gpu.py \
        --blender /content/blender/blender \
        --preset debug --stages 7 \
        --mount-drive --drive-root "{DRIVE_ROOT}" \
        --resume --strict-render


## 5. Preview frames + brightness/coverage check

A healthy frame has real spatial structure (std well above 0) and is not almost
entirely white. If you see `LOOKS BLANK` here or in the render log, the geometry
was off-frame / over-exposed.


In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
from IPython.display import display

rgb_dir = Path('examples/synthetic_floor_7stage/output/blender_renders/stage_07/rgb')
pngs = sorted(rgb_dir.glob('frame_*.png'))
print(f'{len(pngs)} frame(s) rendered')
if pngs:
    mid = pngs[len(pngs)//2]
    arr = np.asarray(Image.open(mid).convert('RGB'), dtype=np.float32)
    lum = arr.mean(axis=2)
    white = float((lum > 250).mean())
    print(f'{mid.name}: mean={lum.mean():.1f} std={lum.std():.2f} white_frac={white:.3f}')
    print('Looks OK' if (white < 0.97 and lum.std() > 3.0) else 'LOOKS BLANK - check render log')
    display(Image.open(mid))


## 6. Full 7-stage render - balanced preset (resumable)

Safe to re-run after a disconnect: completed stages are pulled from Drive and
skipped. Increase `--frames` for longer clips (e.g. 900 ~= 30s at 30fps).


In [ ]:
!PYTHONPATH=examples/synthetic_floor_7stage/src \
    python3 examples/synthetic_floor_7stage/scripts/run_blender_gpu.py \
        --blender /content/blender/blender \
        --preset balanced --frames 120 --device OPTIX \
        --mount-drive --drive-root "{DRIVE_ROOT}" \
        --resume


## 7. Resume status (portable run-state manifest)

`run_state_blender_gpu.json` is one file summarising every stage's status. It
lives on Drive, so copying the run folder to another machine/account lets you
resume from exactly here.


In [ ]:
import json
from pathlib import Path
rs = Path('examples/synthetic_floor_7stage/output/manifests/run_state_blender_gpu.json')
if rs.exists():
    body = json.loads(rs.read_text())
    print('run_id:', body.get('run_id'), '| in_progress:', body.get('in_progress'))
    for sid, st in sorted(body.get('stages', {}).items()):
        dm = st.get('done_marker') or {}
        print(f"  stage {sid}: complete={st['complete']} frames={dm.get('rgb_frame_count')} quality_ok={(dm.get('frame_quality') or {}).get('ok')}")
else:
    print('No run-state yet - run a render first.')


## 8. Inspect the dataset manifest + camera path


In [ ]:
import json
from pathlib import Path
m = Path('examples/synthetic_floor_7stage/output/manifests/manifest_blender_gpu.json')
if m.exists():
    man = json.loads(m.read_text())
    print('manifest stages:', list((man.get('stage_files') or {}).keys()))
cam = Path('examples/synthetic_floor_7stage/output/blender_renders/stage_07/camera_path.json')
if cam.exists():
    c = json.loads(cam.read_text())
    print('camera frames:', len(c['frames']), '| fps:', c['fps'], '| res:', c['intrinsics']['width_px'], 'x', c['intrinsics']['height_px'])


## 9. Download the rendered videos


In [ ]:
from pathlib import Path
from IPython.display import FileLink, display
for mp4 in sorted(Path('examples/synthetic_floor_7stage/output/video').glob('*.mp4')):
    print(f'{mp4.name} ({mp4.stat().st_size/1024/1024:.1f} MB)')
    display(FileLink(str(mp4)))


## 10. Resilience & resume - how it works

- **Everything is on Drive.** Outputs mirror to
  `MyDrive/MyCon_Colab/synthetic_floor_7stage/<RUN_NAME>/output/` after every
  stage and via a background daemon, so a Colab crash costs at most the current
  stage's wall-clock time.
- **Resume after a disconnect:** re-run cells 1-4 (or 6) with the same
  `RUN_NAME`. The runner pulls prior outputs from Drive, then `--resume` skips
  finished stages and continues.
- **Resume on another device / Drive account:** share or copy the run folder to
  the new Drive, set the same `RUN_NAME`, and run. The `.done` markers + the
  `run_state_blender_gpu.json` manifest travel with the folder.
- **Stale Drive mount** after a reconnect is auto-detected and remounted by the
  sync layer.
- **Blank-frame guard:** `--strict-render` fails fast if a frame is essentially
  all-white / structureless (the bug this notebook fixes).
